# Streaming LenDB

Plots of the runs that trained on the whole LenDB pool by streaming it --
`results/LenDB-full-*.json`, written by `scripts/stream_full.py` -- against the
65,536-row runs in `results/LenDB-6205faa.json` that every earlier LenDB
comparison used. Nothing here trains anything; it reads the result files, so it
runs in a second and can be re-run after every experiment.

The pool is 975,291 rows. As raw series that is 6.3 GB against 7.9 GB of host
RAM, and 58 GB once QUANT expands it to 14,940 columns, so no run ever holds it.
Validation and test rows are byte-identical to the 65,536-row runs, so the only
thing changing across these curves is how much training data the model saw.

The question the plots are built around: **what does streaming buy, and what
does it cost?** Sections 1 and 2 are the cost, 3 and 4 are what it bought.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D

RESULTS = Path("../results")

# Filenames carry the commit that wrote them, so the streamed HashBoost and
# XGBoost runs live in different files even though they share a split.
FILES = [
    RESULTS / "LenDB-6205faa.json",  # 65,536 rows; the reference every LenDB run used
    RESULTS / "LenDB-full-96c9e18.json",  # streamed HashBoost, 65,536 and 975,291
    RESULTS / "LenDB-full-ba9dd3f.json",  # streamed XGBoost, 245,760 and 975,291
]

[p.name for p in FILES if p.exists()]

## Style

The palette from `sweeps.ipynb`. One hue per *engine* and never per rank:
HashBoost is blue and XGBoost orange in every figure below, so a colour means
the same thing throughout and a figure that drops a run does not repaint the
survivors. Pool size is a secondary encoding -- filled marker and solid line for
the full pool, hollow and dashed for a subset -- never a third hue.

Only the two hues that clear 3:1 against the surface carry marks here; the rest
of the palette stays available for figures that need more identities.

In [ ]:
SURFACE = "#fcfcfb"
INK = "#0b0b0b"
INK_2 = "#52514e"
MUTED = "#8a8880"
GRID = "#e6e5e1"

BLUE = "#2a78d6"
ORANGE = "#eb6834"
AQUA = "#1baf7a"
YELLOW = "#eda100"
MAGENTA = "#e87ba4"
VIOLET = "#4a3aa7"
SERIES = [BLUE, ORANGE, AQUA, YELLOW, MAGENTA, VIOLET]

ENGINE = {"hashboost": BLUE, "xgboost": ORANGE}

plt.rcParams.update(
    {
        "figure.facecolor": SURFACE,
        "axes.facecolor": SURFACE,
        "savefig.facecolor": SURFACE,
        "axes.edgecolor": GRID,
        "axes.labelcolor": INK_2,
        "axes.titlecolor": INK,
        "axes.titlesize": 11,
        "axes.titleweight": "bold",
        "axes.titlelocation": "left",
        "axes.labelsize": 9,
        "axes.grid": True,
        "axes.axisbelow": True,
        "grid.color": GRID,
        "grid.linewidth": 0.8,
        "grid.linestyle": "-",
        "xtick.color": INK_2,
        "ytick.color": INK_2,
        "xtick.labelsize": 8.5,
        "ytick.labelsize": 8.5,
        "legend.frameon": False,
        "legend.fontsize": 8.5,
        "lines.linewidth": 2.0,
        "figure.dpi": 120,
    }
)


def finish(ax, title=None, note=None, xlabel=None, ylabel=None):
    """Title, optional sub-note, recessive chrome."""

    if title:
        ax.set_title(title, pad=16 if note else 8)
    if note:
        ax.text(
            0.0,
            1.02,
            note,
            transform=ax.transAxes,
            fontsize=8.5,
            color=INK_2,
            va="bottom",
        )
    if xlabel:
        ax.set_xlabel(xlabel)
    if ylabel:
        ax.set_ylabel(ylabel)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    return ax


def rows(n):
    """65,536 -> '66k'. Row counts are the x axis of half these figures."""

    return f"{n / 1e6:.2f}M" if n >= 1e6 else f"{n / 1e3:.0f}k"

## Loading

Two tidy frames: one row per run (`runs`), one per evaluation point (`curves`).

The files disagree with each other in three places, and the loader is where that
is reconciled rather than in each plot. All three are noted in its docstring;
the one that would silently corrupt a figure is the x axis, where XGBoost counts
from zero and HashBoost records a round count.

In [ ]:
def load_runs(paths):
    """-> (one row per run, one row per evaluation point).

    Three things the files do not agree on:

    * **The x axis.** XGBoost's `evals_result` is indexed from zero; HashBoost's
      curve records a round *count*. Shifting the former by one makes both mean
      "rounds completed", which is what every figure here plots against.
    * **`timings`.** `stream_full.py` labels each entry with a `phase`
      (`build`/`train`); the notebook that wrote the reference file appended one
      entry per rerun of its cell, so there the last entry is the run.
    * **`split.n_tr`.** That block describes whichever run was written last, so
      a run's own `n_tr` wins wherever it records one.
    """

    runs, curves = [], []

    for path in paths:
        if not path.exists():
            continue

        blob = json.loads(path.read_text())
        features = blob["transform"]["num_features"]

        for name, entry in blob["models"].items():
            engine = "hashboost" if name.startswith("hashboost") else "xgboost"
            memory = entry.get("memory") or {}

            phased = [t for t in entry["timings"] if t.get("phase")]
            build = next((t["wall_s"] for t in phased if t["phase"] == "build"), 0.0)
            train = next(
                (t["wall_s"] for t in phased if t["phase"] == "train"),
                entry["timings"][-1]["wall_s"],
            )

            shift = 1 if engine == "xgboost" else 0
            tr = entry["results"]["tr"]["merror"]
            va = entry["results"]["va"]["merror"]
            n_rounds = entry.get("rounds") or int(va["x"][-1]) + shift

            runs.append(
                {
                    "run": name,
                    "commit": path.stem.split("-")[-1],
                    "engine": engine,
                    "n_tr": entry.get("n_tr", blob["split"]["n_tr"]),
                    "rounds": n_rounds,
                    "tr_final": tr["y"][-1],
                    "va_final": va["y"][-1],
                    "va_best": min(va["y"]),
                    "build_s": build,
                    "train_s": train,
                    "s_per_round": train / n_rounds,
                    "peak_gpu_gb": memory.get("device_peak_mb", np.nan) / 1024,
                    "host_rss_gb": memory.get("host_peak_rss_mb", np.nan) / 1024,
                    "cache_gb": memory.get("cache_gb", np.nan),
                    "batch_size": memory.get("batch_size", blob["split"]["batch_size"]),
                    "features": features,
                    "resident": memory.get("resident", "whole matrix, in memory"),
                }
            )

            for split in ("tr", "va"):
                c = entry["results"][split]["merror"]
                for x, y in zip(c["x"], c["y"]):
                    curves.append(
                        {
                            "run": name,
                            "engine": engine,
                            "n_tr": runs[-1]["n_tr"],
                            "split": split,
                            "round": x + shift,
                            "error": y,
                        }
                    )

    return (
        pd.DataFrame(runs)
        .sort_values(["engine", "n_tr", "rounds"])
        .reset_index(drop=True),
        pd.DataFrame(curves),
    )


runs, curves = load_runs(FILES)

# The full pool is whatever the largest run streamed; everything else is a subset.
FULL = runs["n_tr"].max()
runs["full_pool"] = runs["n_tr"] == FULL

print(
    f"{len(runs)} runs over {runs['n_tr'].nunique()} pool sizes; full pool = {FULL:,} rows"
)

### The table

Every run. `resident` is what the method must have available to take one step --
the thing section 1 is about.

In [ ]:
table = runs.assign(
    pool=[f"{n:,}" for n in runs["n_tr"]],
    error=[f"{b:.4f}" for b in runs["va_best"]],
    train=[f"{t:.4f}" for t in runs["tr_final"]],
    wall=[f"{b + t:.0f}s" for b, t in zip(runs["build_s"], runs["train_s"])],
)[
    [
        "run",
        "commit",
        "pool",
        "rounds",
        "train",
        "error",
        "wall",
        "s_per_round",
        "peak_gpu_gb",
        "cache_gb",
        "resident",
    ]
]

table.style.format(
    {"s_per_round": "{:.2f}s", "peak_gpu_gb": "{:.2f} GB", "cache_gb": "{:.1f} GB"},
    na_rep="--",
).hide(axis="index")

## 1. Peak memory does not move with the data

Read the left panel first, and notice that it is deliberately boring: **peak GPU
memory is flat in both engines.** HashBoost took the same 2.51 GB for 65,536
rows and for 975,291. XGBoost sat near 5.9 GB at every size. Neither number
knows how large the training set is, because what dominates on the card is the
histogram and the leaf tables, which scale with features, bins and classes.

The right panel is where the two part company. It is the training matrix each
method must have *available* to take one step. HashBoost needs the batch it is
fitting and nothing else, so its line is flat at 0.24 GB by construction --
`fit_batch` folds a batch in and frees it, and the pool size never enters.
XGBoost needs the whole binned matrix every round, so streaming only decides
where that matrix lives: on the card at 65,536 rows, and on disk once past the
8.6 GB of card and 7.9 GB of host RAM that the two rules mark.

The ellpack measures almost exactly **one byte per element**, worth recording
because a global symbol space over 14,940 x 256 bins would imply 22 bits and a
40 GB matrix. It is 14.6 GB.

In [ ]:
# One byte per element, measured rather than assumed: the two external-memory
# runs wrote their ellpack to disk, so its size is a fact. The in-memory run's
# ellpack never left the card and is derived from that rate (hollow marker).
measured = runs.dropna(subset=["cache_gb"])
per_element = (
    measured["cache_gb"] * 1e9 / (measured["n_tr"] * measured["features"])
).mean()

runs["resident_gb"] = np.where(
    runs["engine"] == "hashboost",
    runs["batch_size"] * runs["features"] * 4 / 1e9,  # one batch of float32 features
    runs["n_tr"] * runs["features"] * per_element / 1e9,  # the whole binned matrix
)

CARD_GB, HOST_GB = 8.585, 7.886

fig, axes = plt.subplots(1, 2, figsize=(10.6, 4.3), gridspec_kw={"wspace": 0.26})

for ax, column, measured_only in (
    (axes[0], "peak_gpu_gb", True),
    (axes[1], "resident_gb", False),
):
    for engine, g in runs.groupby("engine"):
        g = g.dropna(subset=[column]).sort_values("n_tr")
        if measured_only:
            g = g.drop_duplicates("n_tr", keep="last")
        if g.empty:
            continue

        ax.plot(g["n_tr"], g[column], color=ENGINE[engine], zorder=3, label=engine)
        for _, r in g.iterrows():
            # filled = weighed on disk, hollow = computed from bytes/element
            computed = column == "resident_gb" and np.isnan(r["cache_gb"])
            ax.plot(
                r["n_tr"],
                r[column],
                marker="o",
                ms=8,
                mfc=SURFACE if computed else ENGINE[engine],
                mec=ENGINE[engine],
                mew=2,
                zorder=4,
            )

    ax.set_xscale("log")
    ax.set_xticks(sorted(runs["n_tr"].unique()))
    ax.set_xticklabels([rows(n) for n in sorted(runs["n_tr"].unique())])
    ax.minorticks_off()

axes[0].set_ylim(0, CARD_GB * 1.08)
axes[0].axhline(CARD_GB, color=MUTED, lw=1.0, ls=(0, (4, 3)), zorder=1)
axes[0].annotate(
    "the card, 8.6 GB",
    (0.99, CARD_GB),
    xycoords=("axes fraction", "data"),
    ha="right",
    va="bottom",
    fontsize=7.5,
    color=MUTED,
)

for engine, g in runs.dropna(subset=["peak_gpu_gb"]).groupby("engine"):
    last = g.sort_values("n_tr").iloc[-1]
    axes[0].annotate(
        f"{last['peak_gpu_gb']:.2f} GB",
        (last["n_tr"], last["peak_gpu_gb"]),
        textcoords="offset points",
        xytext=(-6, 10),
        ha="right",
        fontsize=8,
        color=ENGINE[engine],
    )

axes[1].set_yscale("log")
axes[1].set_ylim(bottom=0.15)
for y, text in ((CARD_GB, "the card, 8.6 GB"), (HOST_GB, "host RAM, 7.9 GB")):
    axes[1].axhline(y, color=MUTED, lw=1.0, ls=(0, (4, 3)), zorder=1)
axes[1].annotate(
    "card 8.6 GB / host RAM 7.9 GB",
    (0.99, CARD_GB * 1.06),
    xycoords=("axes fraction", "data"),
    ha="right",
    va="bottom",
    fontsize=7.5,
    color=MUTED,
)

for _, r in runs.iterrows():
    axes[1].annotate(
        f"{r['resident_gb']:.2f}"
        if r["resident_gb"] < 1
        else f"{r['resident_gb']:.1f}",
        (r["n_tr"], r["resident_gb"]),
        textcoords="offset points",
        xytext=(0, 10),
        ha="center",
        fontsize=8,
        color=ENGINE[r["engine"]],
    )

axes[0].legend(loc="upper left")
finish(
    axes[0],
    "Peak GPU memory",
    "flat in both: histograms and leaf tables, not the data",
    xlabel="training rows",
    ylabel="peak device memory (GB)",
)
finish(
    axes[1],
    "The matrix a step needs available",
    f"XGBoost's ellpack measures {per_element:.2f} bytes/element; hollow = derived",
    xlabel="training rows",
    ylabel="GB (log)",
)
plt.show()

## 2. What a round costs

XGBoost's cost per round is linear in rows: 0.7s at 65,536 where the matrix is
on the card, 16s at 245,760, and 116s at 975,291 where every round pages the
ellpack back off disk. That is **171x the in-memory cost for 15x the data**, and
it is the price of the right-hand panel above.

HashBoost's is not linear in rows. Its per-round work is proportional to the
number of *rounds* -- every batch updates the buckets of all existing rounds --
so streaming 15x the data moved it only 0.19s to 0.47s at matched rounds, and
that 2.4x is streaming I/O rather than the model update.

The hollow HashBoost point at 65k is the notebook's own 800-round run on the
same rows. It is cheaper per round than the 1,200-round run beside it for
exactly the reason above, which is why the matched-round pair is the one to
read.

In [ ]:
fig, ax = plt.subplots(figsize=(8.0, 4.4))

for engine, g in runs.groupby("engine"):
    g = g.sort_values(["n_tr", "rounds"])
    # one line through the comparable runs; the shorter reference run sits off it
    spine = g.drop_duplicates("n_tr", keep="last")
    ax.plot(
        spine["n_tr"],
        spine["s_per_round"],
        color=ENGINE[engine],
        zorder=3,
        label=engine,
    )

    for _, r in g.iterrows():
        on_spine = r["run"] in set(spine["run"])
        # the reference run is context, so it is grey rather than a paler version
        # of the engine's hue -- nothing off the line should read as a series
        colour = ENGINE[engine] if on_spine else MUTED
        ax.plot(r["n_tr"], r["s_per_round"], marker="o", ms=8, color=colour, zorder=4)
        ax.annotate(
            f"{r['s_per_round']:.2f}s"
            + ("" if on_spine else f"  ({r['rounds']} rounds)"),
            (r["n_tr"], r["s_per_round"]),
            textcoords="offset points",
            xytext=(10, 3) if on_spine else (10, -13),
            fontsize=8,
            color=colour,
        )

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xticks(sorted(runs["n_tr"].unique()))
ax.set_xticklabels([rows(n) for n in sorted(runs["n_tr"].unique())])
ax.minorticks_off()
ax.set_xlim(right=runs["n_tr"].max() * 1.9)
ax.legend(loc="upper left")

finish(
    ax,
    "Seconds per boosting round",
    "grey = the 800-round reference run, cheaper per round because it has fewer rounds",
    xlabel="training rows",
    ylabel="seconds per round (log)",
)
plt.show()

## 3. What the extra data bought, per round

Small multiples rather than six lines on one axis, because the two engines are
not racing: HashBoost is GPU time over many cheap rounds, XGBoost is a handful
of expensive ones.

At matched rounds the **small pool wins** -- 0.0532 against 0.0691 over ~1,200
rounds, on the same gradient steps, the same number of samples processed, and
identical model capacity. That reads like more data making things worse. It is
not, and section 4 is why.

The two full-pool XGBoost runs are three rounds each: enough to measure what a
round costs, nowhere near enough to converge. Read them as cost probes, not as
scores.

In [ ]:
va = curves[curves["split"] == "va"]

fig, axes = plt.subplots(
    1, 2, figsize=(10.6, 4.4), sharey=True, gridspec_kw={"wspace": 0.08}
)

for ax, engine in zip(axes, ("hashboost", "xgboost")):
    g = runs[runs["engine"] == engine].sort_values("n_tr")

    # the runs in a panel end within a few thousandths of each other, so spread
    # the end labels by where each finishes rather than by drawing order
    spread = np.linspace(-11, 11, len(g))
    rank = {run: i for i, run in enumerate(g.sort_values("va_final")["run"])}

    for _, r in g.iterrows():
        sub = va[va["run"] == r["run"]].sort_values("round")
        style = "-" if r["full_pool"] else (0, (5, 2))
        ax.plot(
            sub["round"],
            sub["error"],
            color=ENGINE[engine],
            ls=style,
            alpha=1.0 if r["full_pool"] else 0.75,
            marker="o" if len(sub) < 8 else None,
            ms=6,
            label=f"{rows(r['n_tr'])} rows, {r['rounds']} rounds",
            zorder=3,
        )
        # the value where the curve actually ends -- for xgboost_975k the best
        # round is not the last one, and labelling the best here would put a
        # number next to a point that does not carry it
        ax.annotate(
            f"{sub['error'].iloc[-1]:.4f}",
            (sub["round"].iloc[-1], sub["error"].iloc[-1]),
            textcoords="offset points",
            xytext=(8, spread[rank[r["run"]]]),
            va="center",
            fontsize=8,
            color=ENGINE[engine],
        )

    ax.set_xscale("log")
    ax.set_xlim(right=ax.get_xlim()[1] * 2.6)
    ax.legend(loc="upper right")

axes[0].set_ylim(0.03, 0.20)
finish(
    axes[0],
    "HashBoost",
    "solid = full pool, dashed = a subset",
    xlabel="rounds completed (log)",
    ylabel="validation error",
)
finish(
    axes[1],
    "XGBoost",
    "the full-pool runs are 3 rounds",
    xlabel="rounds completed (log)",
)
plt.show()

## 4. Why the small pool wins: it memorised

Each row is one run, drawn from its training error to its validation error. The
gap is the diagnosis.

The 65,536-row HashBoost reaches **tr = 0.0049** -- it has essentially memorised
its training set -- and still generalises to 0.0532. The full-pool run sits at
tr = 0.0608 against va = 0.0691, train and validation almost touching, which is
the signature of a model that has not finished fitting. 1,200 hashes are enough
to memorise 65k rows and nowhere near enough for 975k.

So the full-pool run is **capacity-bound, not data-bound**, and its curve in
section 3 was still descending when it stopped. Matching the reference's 50
passes over 975,291 rows is ~9,000 rounds, and since per-batch cost grows
linearly with rounds that is roughly 60x the compute spent here. That quadratic,
not memory, is where HashBoost's scaling actually binds.

The three-round XGBoost runs are shown for completeness; a model that barely
trained has a small gap for reasons that have nothing to do with capacity.

In [ ]:
order = runs.sort_values(["engine", "n_tr", "rounds"]).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(8.4, 0.62 * len(order) + 1.8))

for i, r in order.iterrows():
    colour = ENGINE[r["engine"]]
    thin = r["rounds"] < 8  # barely trained; the gap is not a capacity statement

    ax.plot(
        [r["tr_final"], r["va_final"]],
        [i, i],
        color=colour if not thin else MUTED,
        lw=2.0,
        alpha=0.45,
        zorder=2,
    )
    ax.scatter(r["tr_final"], i, s=64, color=SURFACE, edgecolor=colour, lw=2, zorder=3)
    ax.scatter(r["va_final"], i, s=64, color=colour, zorder=3)

    ax.annotate(
        f"gap {r['va_final'] - r['tr_final']:+.4f}",
        (0.995, i),
        xycoords=("axes fraction", "data"),
        ha="right",
        va="center",
        fontsize=8,
        color=MUTED if thin else INK_2,
    )

ax.set_yticks(
    range(len(order)),
    [
        f"{r['run']}\n{rows(r['n_tr'])} rows, {r['rounds']} rounds"
        for _, r in order.iterrows()
    ],
    fontsize=8,
)
ax.set_ylim(-0.7, len(order) - 0.3)
ax.set_xlim(-0.004, 0.115)
ax.invert_yaxis()
ax.grid(axis="y", visible=False)
ax.legend(
    handles=[
        Line2D(
            [], [], marker="o", ls="none", mfc=SURFACE, mec=INK_2, mew=2, label="train"
        ),
        Line2D([], [], marker="o", ls="none", color=INK_2, label="validation"),
    ],
    loc="lower right",
    bbox_to_anchor=(1.0, 1.0),
    ncol=2,
)

finish(
    ax,
    "Training error to validation error",
    "a wide gap is memorisation; train ~= validation is a model still underfitting",
    xlabel="misclassification rate",
)
plt.show()

## 5. Where the wall time goes

Building the ellpack is a fixed cost paid once: 353s for the full pool, three
passes over 975,291 rows to sketch the quantiles and write 14.6 GB. Training is
then linear in rounds at 116s each. At three rounds the two are about equal,
which flatters the build -- at the 179 rounds the 65,536-row run needed, the
training bar would be **5.8 hours** against the same 353s.

HashBoost has no build phase at all. The QUANT transform is applied per batch on
the way past, which is why its bar is one colour.

In [ ]:
order = runs.sort_values(["engine", "n_tr", "rounds"]).reset_index(drop=True)
y = np.arange(len(order))

fig, ax = plt.subplots(figsize=(8.4, 0.55 * len(order) + 1.8))

ax.barh(
    y,
    order["build_s"],
    color=MUTED,
    alpha=0.55,
    height=0.6,
    label="build the matrix",
    zorder=3,
)
ax.barh(
    y,
    order["train_s"],
    left=order["build_s"],
    color=[ENGINE[e] for e in order["engine"]],
    height=0.6,
    label="train",
    zorder=3,
)

for i, r in order.iterrows():
    total = r["build_s"] + r["train_s"]
    ax.annotate(
        f"{total:.0f}s   {r['rounds']} rounds at {r['s_per_round']:.2f}s",
        (total, i),
        textcoords="offset points",
        xytext=(7, 0),
        va="center",
        fontsize=8,
        color=INK_2,
    )

ax.set_yticks(
    y, [f"{r['run']}\n{rows(r['n_tr'])} rows" for _, r in order.iterrows()], fontsize=8
)
ax.set_ylim(len(order) - 0.4, -0.6)
ax.set_xlim(right=(order["build_s"] + order["train_s"]).max() * 1.55)
ax.grid(axis="y", visible=False)
ax.legend(
    handles=[
        Line2D(
            [], [], marker="s", ls="none", color=MUTED, alpha=0.55, ms=9, label="build"
        ),
        Line2D(
            [], [], marker="s", ls="none", color=BLUE, ms=9, label="train (HashBoost)"
        ),
        Line2D(
            [], [], marker="s", ls="none", color=ORANGE, ms=9, label="train (XGBoost)"
        ),
    ],
    loc="upper right",
)

finish(
    ax,
    "Wall clock, build against train",
    "HashBoost has no build phase; the transform rides along with each batch",
    xlabel="seconds",
)
plt.show()